# Notebook 04: Demand Forecasting (Prophet)
## Retail Demand Forecasting & Inventory Optimization

**Objective:** Use Facebook Prophet to forecast demand at 30 and 90-day horizons for categories and stores.

**Inputs:** `Data/processed/` parquet files (no database needed)

**Outputs:** `category_forecasts.parquet`, `store_forecasts.parquet`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from prophet import Prophet
from sklearn.metrics import mean_absolute_error, mean_squared_error
from pathlib import Path
import logging, warnings
warnings.filterwarnings('ignore')
logging.getLogger('prophet').setLevel(logging.WARNING)
logging.getLogger('cmdstanpy').setLevel(logging.WARNING)

DATA_PROC = Path('../Data/processed')
IMAGES    = Path('../images')
IMAGES.mkdir(exist_ok=True)

plt.rcParams.update({
    'figure.facecolor': '#0f172a', 'axes.facecolor': '#1e293b',
    'axes.edgecolor': '#334155', 'text.color': '#f1f5f9',
    'axes.labelcolor': '#f1f5f9', 'xtick.color': '#94a3b8',
    'ytick.color': '#94a3b8', 'grid.color': '#334155',
})
PALETTE = ['#38bdf8', '#34d399', '#fb923c', '#818cf8', '#f472b6']
CAT_CLR = {'FOODS': '#34d399', 'HOUSEHOLD': '#38bdf8', 'HOBBIES': '#fb923c'}

print('Prophet forecasting environment ready')

## Step 1: Load Data

In [ ]:
daily_agg = pd.read_parquet(DATA_PROC / 'daily_aggregated.parquet')
daily_agg['date'] = pd.to_datetime(daily_agg['date'])

df_sample = pd.read_parquet(DATA_PROC / 'sales_enriched_sample.parquet')
df_sample['date'] = pd.to_datetime(df_sample['date'])

print(f'Daily agg : {len(daily_agg):,} rows | {daily_agg.date.min().date()} to {daily_agg.date.max().date()}')
print(f'Sample SKU: {len(df_sample):,} rows | {df_sample.product_id.nunique()} SKUs')

In [ ]:
# Build M5 holiday calendar from parquet (no DB needed)
cal = pd.read_parquet(DATA_PROC / 'calendar_clean.parquet')
cal['date'] = pd.to_datetime(cal['date'])

holidays = (
    cal[cal['event_name_1'].notna()][['date', 'event_name_1']]
    .rename(columns={'date': 'ds', 'event_name_1': 'holiday'})
    .copy()
)
holidays['lower_window'] = -1
holidays['upper_window'] = 1

print(f'Holidays loaded: {len(holidays)} event days ({holidays.holiday.nunique()} unique events)')

## Step 2: Helper Functions

In [ ]:
def fit_prophet(series, horizon_days=90, holidays=None):
    """Fit Prophet and return (model, forecast)."""
    df_p = pd.DataFrame({'ds': series.index, 'y': series.values})
    m = Prophet(
        yearly_seasonality=True,
        weekly_seasonality=True,
        daily_seasonality=False,
        changepoint_prior_scale=0.05,
        seasonality_prior_scale=10,
        holidays=holidays,
        interval_width=0.80
    )
    m.fit(df_p)
    future   = m.make_future_dataframe(periods=horizon_days)
    forecast = m.predict(future)
    return m, forecast


def evaluate(actual, predicted):
    a, p = np.array(actual), np.array(predicted)
    mask = a > 0
    mae  = mean_absolute_error(a, p)
    rmse = np.sqrt(mean_squared_error(a, p))
    mape = np.mean(np.abs((a[mask] - p[mask]) / a[mask])) * 100
    return {'MAE': round(mae, 1), 'RMSE': round(rmse, 1), 'MAPE': round(mape, 2)}


def plot_forecast(historical, forecast, title, save_path=None):
    fig, ax = plt.subplots(figsize=(16, 5))
    ax.set_facecolor('#1e293b')
    fig.patch.set_facecolor('#0f172a')
    ax.fill_between(forecast['ds'], forecast['yhat_lower'], forecast['yhat_upper'],
                    color=PALETTE[0], alpha=0.15, label='80% CI')
    ax.plot(forecast['ds'], forecast['yhat'], color=PALETTE[0], linewidth=2, label='Forecast')
    ax.plot(historical.index, historical.values, color=PALETTE[1],
            linewidth=1.2, alpha=0.85, label='Actual')
    ax.axvline(historical.index.max(), color='white', linestyle='--', linewidth=1, alpha=0.5)
    ax.set_title(title, color='white', fontsize=13)
    ax.set_ylabel('Units Sold', color='#94a3b8')
    ax.legend(loc='upper left')
    ax.grid(True, alpha=0.3)
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight', facecolor='#0f172a')
    plt.show()

print('Helper functions ready')

## Step 3: Category-Level Forecasting (90-Day)

In [ ]:
category_forecasts = {}
category_metrics   = {}
HORIZON = 90

for cat in ['FOODS', 'HOUSEHOLD', 'HOBBIES']:
    print(f'\nForecasting: {cat} ...')

    cat_daily = (
        daily_agg[daily_agg['cat_id'] == cat]
        .groupby('date')['total_units'].sum()
        .asfreq('D').fillna(0)
    )

    train = cat_daily.iloc[:-HORIZON]
    test  = cat_daily.iloc[-HORIZON:]

    m, forecast = fit_prophet(train, horizon_days=HORIZON, holidays=holidays)

    # Evaluate on held-out test period
    forecast_test = forecast[forecast['ds'].isin(test.index)]['yhat'].values
    metrics = evaluate(test.values, forecast_test)
    print(f'  MAE={metrics["MAE"]:,.0f} | RMSE={metrics["RMSE"]:,.0f} | MAPE={metrics["MAPE"]:.1f}%')

    category_forecasts[cat] = (m, forecast, cat_daily)
    category_metrics[cat]   = metrics

    plot_forecast(
        cat_daily, forecast,
        title=f'{cat} - 90-Day Demand Forecast',
        save_path=IMAGES / f'04_forecast_{cat.lower()}.png'
    )

print('\nCategory forecasts complete')

In [ ]:
metrics_df = pd.DataFrame(category_metrics).T.reset_index()
metrics_df.columns = ['Category', 'MAE', 'RMSE', 'MAPE (%)']
metrics_df['Accuracy'] = (100 - metrics_df['MAPE (%)']).round(1)
print('Category Forecast Performance (90-Day):')
display(metrics_df)

## Step 4: Store-Level Forecasting (30-Day)

In [ ]:
store_forecasts = {}
store_metrics   = {}
HORIZON = 30

for store in sorted(daily_agg['store_id'].unique()):
    store_daily = (
        daily_agg[daily_agg['store_id'] == store]
        .groupby('date')['total_units'].sum()
        .asfreq('D').fillna(0)
    )
    train = store_daily.iloc[:-HORIZON]
    test  = store_daily.iloc[-HORIZON:]

    m, forecast = fit_prophet(train, horizon_days=HORIZON, holidays=holidays)
    metrics = evaluate(test.values, forecast[forecast['ds'].isin(test.index)]['yhat'].values)

    store_forecasts[store] = (m, forecast, store_daily)
    store_metrics[store]   = metrics
    print(f'  {store}: MAPE={metrics["MAPE"]:.1f}%')

store_metrics_df = pd.DataFrame(store_metrics).T.reset_index()
store_metrics_df.columns = ['Store', 'MAE', 'RMSE', 'MAPE (%)']
print('\nStore Forecast Performance (30-Day):')
display(store_metrics_df.sort_values('MAPE (%)'))

## Step 5: Prophet Components (FOODS)

In [ ]:
m_foods, forecast_foods, _ = category_forecasts['FOODS']
fig = m_foods.plot_components(forecast_foods)
fig.patch.set_facecolor('#0f172a')
for ax in fig.get_axes():
    ax.set_facecolor('#1e293b')
    ax.tick_params(colors='white')
    ax.title.set_color('white')
plt.suptitle('Prophet Components - FOODS Category', color='white', fontsize=12, y=1.01)
plt.savefig(IMAGES / '04_prophet_components.png', dpi=150, bbox_inches='tight', facecolor='#0f172a')
plt.show()

## Step 6: Save Forecasts to Parquet

In [ ]:
# Category forecasts
cat_rows = []
for cat, (m, fc, _) in category_forecasts.items():
    fut = fc[['ds','yhat','yhat_lower','yhat_upper']].tail(90).copy()
    fut['cat_id'] = cat
    cat_rows.append(fut)

cat_fc_df = pd.concat(cat_rows, ignore_index=True)
cat_fc_df.rename(columns={'ds':'forecast_date','yhat':'predicted_demand',
                           'yhat_lower':'lower_ci','yhat_upper':'upper_ci'}, inplace=True)
cat_fc_df.to_parquet(DATA_PROC / 'category_forecasts.parquet', index=False)
print(f'Saved category_forecasts.parquet: {len(cat_fc_df)} rows')

# Store forecasts
store_rows = []
for store, (m, fc, _) in store_forecasts.items():
    fut = fc[['ds','yhat','yhat_lower','yhat_upper']].tail(30).copy()
    fut['store_id'] = store
    store_rows.append(fut)

store_fc_df = pd.concat(store_rows, ignore_index=True)
store_fc_df.rename(columns={'ds':'forecast_date','yhat':'predicted_demand',
                              'yhat_lower':'lower_ci','yhat_upper':'upper_ci'}, inplace=True)
store_fc_df.to_parquet(DATA_PROC / 'store_forecasts.parquet', index=False)
print(f'Saved store_forecasts.parquet   : {len(store_fc_df)} rows')

print('\nAll forecasts saved. Proceed to Notebook 05 (Inventory Optimization).')